# IndoBERTweet fine-tuning with class-weighted loss, then CatBoost on the fine-tuned embeddings

In [ ]:
DATA_DIR = "../data"

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/penyisihan-bdc-2024/keywords_prob.xlsx
/kaggle/input/penyisihan-bdc-2024/leakage - leakage.csv
/kaggle/input/penyisihan-bdc-2024/with_embed_full.pkl
/kaggle/input/penyisihan-bdc-2024/sample_submission.csv
/kaggle/input/penyisihan-bdc-2024/train_cleaned_v3_keywords.csv
/kaggle/input/penyisihan-bdc-2024/val_v2.csv
/kaggle/input/penyisihan-bdc-2024/trainaug_v2.csv
/kaggle/input/penyisihan-bdc-2024/train_cleaned_v2_translated.csv
/kaggle/input/penyisihan-bdc-2024/trainaug_indo.csv
/kaggle/input/penyisihan-bdc-2024/train_id.csv
/kaggle/input/penyisihan-bdc-2024/train_fix - train.csv
/kaggle/input/penyisihan-bdc-2024/val.csv
/kaggle/input/penyisihan-bdc-2024/val_indo.csv
/kaggle/input/penyisihan-bdc-2024/train_cleaned_v2_keywords.csv
/kaggle/input/penyisihan-bdc-2024/test_bdc.xlsx
/kaggle/input/penyisihan-bdc-2024/trainaug.csv
/kaggle/input/penyisihan-bdc-2024/train.csv
/kaggle/input/penyisihan-bdc-2024/test.csv
/kaggle/input/penyisihan-bdc-2024/with_embed_test_full.pkl
/kaggle

In [ ]:
import pandas as pd
import numpy as np
import re

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    # text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE\s+(\w+)\]', r'\1', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9%]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
        "org": "orang",
        "pemda": "pemerintah daerah",
        "pemkot": "pemerintah kota",
        "pemprov": "pemerintah provinsi",
        "pemkab": "pemerintah kabupaten",
        "pemilu": "pemilihan umum",
        "pilkada": "pemilihan kepala daerah",
        "pilpres": "pemilihan presiden",
        "pilgub": "pemilihan gubernur",
        "pilbup": "pemilihan bupati",
        "pilwali": "pemilihan wali kota",
        "pilleg": "pemilihan legislatif",
        "pilpres": "pemilihan presiden",
        "ajg" : "bajingan",
        "anjg" : "banjingan",
        "anjir" : "banjingan",
        "anjay" : "banjingan",
        "anjayy" : "banjingan",
        "anjayyy" : "banjingan",
        "ikn" : "ibu kota negara",
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")
    text = re.sub("alam ganjar", 'anaknya ganjar', text, flags=re.IGNORECASE)
    text = re.sub("%", 'persen', text, flags=re.IGNORECASE)


    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', sep=";")

In [ ]:
train_fix_df = pd.read_csv(f'{DATA_DIR}/train_fix - train.csv')

# get index that is true in is_false column
false_idx = train_fix_df['is_false'] == 1

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv', sep=";")
test_df = pd.read_csv(f'{DATA_DIR}/test.csv', sep=";")
train_df.head()
# remove train_df based on false index
train_df = train_df[~false_idx].copy()

/tmp/ipykernel_35/191052703.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  train_df = train_df[~false_idx].copy()


In [ ]:
train_df['text'] = train_df['text'].apply(remove_noise)
train_df['text'] = train_df['text'].apply(normalize_word)
test_df['Text'] = test_df['Text'].apply(remove_noise)
test_df['Text'] = test_df['Text'].apply(normalize_word)
train_df = drop_duplicated(train_df)
train_df.head()

Df size: (4939, 2)
After dropping empty text rows, df size:  (4930, 2)
After dropping literal duplicated rows, df size:  (4295, 2)
After dropping text duplicated rows, df size:  (4186, 2)


,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik


In [ ]:
train_df['text'] = train_df['text'].str.lower()
test_df['Text'] = test_df['Text'].str.lower()

In [ ]:
kw_df = pd.read_excel(f"{DATA_DIR}/keywords_prob.xlsx")
kw_df.head()

,keyword,Demografi,Ekonomi,Geografi,Ideologi,Pertahanan dan Keamanan,Sosial Budaya,Sumber Daya Alam,Politik
0,pendidikan rendah,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,kelompok etnis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,masyarakat lintas etnis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,kaum difabel,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,bugis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Create a label mapping
label_mapping = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}

# Apply the label mapping
train['label'] = train['label'].map(label_mapping)

In [ ]:
# Split the DataFrame into training and validation sets
train_df, val_df = train_test_split(train[['text','label']], test_size=0.15, stratify=train['label'], random_state=666)

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

In [ ]:
import torch
import gc


# Mengosongkan cache CUDA
torch.cuda.empty_cache()

# Mengumpulkan garbage
gc.collect()

# Memeriksa penggunaan memori CUDA
print(f"Memory Reserved: {torch.cuda.memory_reserved()} bytes")
print(f"Memory Allocated: {torch.cuda.memory_allocated()} bytes")
print(f"Max Memory Reserved: {torch.cuda.max_memory_reserved()} bytes")

# Restart kernel/environment jika diperlukan

Memory Reserved: 0 bytes
Memory Allocated: 0 bytes
Max Memory Reserved: 0 bytes


In [ ]:
import numpy as np
from collections import Counter

# Assuming 'dataset' is your dataset and 'train' split contains the labels
labels = dataset['train']['label']
class_counts = Counter(labels)
total_samples = len(labels)

# Calculate weights for each class
class_weights = {cls: total_samples/count for cls, count in class_counts.items()}

# Normalize weights to sum to 1
sum_weights = sum(class_weights.values())
class_weights = {cls: weight/sum_weights for cls, weight in class_weights.items()}

# Convert weights to a list
class_weight_list = [class_weights[i] for i in range(len(class_weights))]

In [ ]:
class_weight_list

[0.004153468211636198,
 0.02102537214948504,
 0.03362711763651614,
 0.03085782559586187,
 0.03085782559586187,
 0.06436601658032537,
 0.19795586231307613,
 0.6171565119172374]

In [ ]:
# Define the model name
model_name = "indolem/indobertweet-base-uncased"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/4250 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobertweet-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define the Focal Loss function
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Custom Trainer class with Focal Loss
class FocalLossTrainer(Trainer):
    def __init__(self, alpha=1, gamma=2, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha
        self.gamma = gamma
        self.loss_fn = FocalLoss(alpha=self.alpha, gamma=self.gamma)

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss = self.loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
import torch
from transformers import Trainer, TrainingArguments
from sklearn.metrics import balanced_accuracy_score

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Convert class weights to a tensor
        class_weights_tensor = torch.tensor(class_weight_list).to(model.parameters().__next__().device)

        # Compute weighted cross-entropy loss
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    return {"balanced_accuracy": balanced_acc}

# Define training arguments with evaluation and saving steps
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="balanced_accuracy",  # Use balanced accuracy as the metric
    greater_is_better=True,  # Set to True since higher balanced accuracy is better
)


# Use the custom trainer for training
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # custom compute_metrics function
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

# Clear CUDA cache if necessary
torch.cuda.empty_cache()

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss,Balanced Accuracy
100,1.557900,1.474377,0.405659
200,1.421400,1.154072,0.507519
300,0.936700,1.037685,0.548725
400,1.323500,0.992643,0.600119
500,0.710700,1.022928,0.527068
600,0.499000,0.993638,0.552124
700,0.756700,0.920955,0.628429
800,0.642000,0.907210,0.630804
900,0.354600,1.038095,0.619750
1000,0.272200,0.934126,0.619892


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead u

Balanced Accuracy: 0.630803790377878


In [ ]:
# rename Text to text from test
test_df.rename(columns={"Text": "text"}, inplace=True)

In [ ]:
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


In [ ]:
predictions = trainer.predict(tokenized_test)

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)

In [ ]:
test_df['Kelas'] = preds

In [ ]:
# Reverse the label mapping
reverse_label_mapping = {v: k for k, v in label_mapping.items()}

# Apply the reverse label mapping
test_df['Kelas'] = test_df['Kelas'].map(reverse_label_mapping)

In [ ]:
test_df['Kelas'].value_counts()

Kelas
Politik                    424
Ideologi                   199
Ekonomi                    198
Pertahanan dan Keamanan     84
Sosial Budaya               60
Sumber Daya Alam            26
Demografi                    9
Name: count, dtype: int64

In [ ]:
train_df['label'].map(reverse_label_mapping).value_counts()

label
Politik                    2526
Sosial Budaya               499
Ideologi                    340
Pertahanan dan Keamanan     340
Ekonomi                     312
Sumber Daya Alam            163
Demografi                    53
Geografi                     17
Name: count, dtype: int64

In [ ]:
test_df.to_csv(f"{DATA_DIR}/BERT_submission2.csv")

# EMBEDDING

In [ ]:
from tqdm import tqdm

model = AutoModelForSequenceClassification.from_pretrained('results/checkpoint-500', num_labels=8)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Function to get embeddings using the fine-tuned model
def get_embeddings(texts, tokenizer, model, max_length=512):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Use the embeddings from the last hidden state (average pooling)
    embeddings = outputs.hidden_states[-1].mean(dim=1).cpu().numpy()
    return embeddings

# Assuming your train dataset is in a pandas DataFrame
train_texts = train['text'].tolist()

# Get embeddings for the train dataset
train_embeddings = []
for text in tqdm(train_texts, desc="Generating embeddings"):
    embedding = get_embeddings([text], tokenizer, model)
    train_embeddings.append(embedding)

train_embeddings = np.vstack(train_embeddings)

# Convert embeddings to DataFrame and add as new features to the original dataset
embedding_df = pd.DataFrame(train_embeddings)
train_dataset_with_embeddings = pd.concat([train, embedding_df], axis=1)

# Print the shape to verify
print(train_dataset_with_embeddings.shape)

(4169, 2542)


In [ ]:
train_dataset_with_embeddings.to_csv(f'{DATA_DIR}/train_dataset_with_embeddings.csv', index=False)

In [ ]:
train_dataset_with_embeddings

In [ ]:
columns_to_remove = [
    'label_pol', 'label_sos', 'label_eko', 'label_aman',
    'label_ide', 'label_sda', 'label_demo', 'label_geo'
]

train_dataset_with_embeddings = train_dataset_with_embeddings.drop(columns=columns_to_remove)

                                                text  label  \
0  kunjungan prabowo ini untuk meresmikan dan men...      5   
1  anies dapat tepuk tangan meriah saat jadi rekt...      0   
2  emng bener sih pendukung 01 ada yang goblok be...      6   
3  sewaktu anies bersikap kritis ke kinerja pak p...      0   
4  anies baswedan harap aparatur sipil negara ter...      0   

   pendidikan rendah  kelompok etnis  masyarakat lintas etnis  kaum difabel  \
0                  0               0                        0             0   
1                  0               0                        0             0   
2                  1               0                        0             0   
3                  0               0                        0             0   
4                  0               0                        0             0   

   bugis  generasi emas  low educated  menengah bawah  ...       758  \
0      0              0             0               0  ...  0.205502   
1 

In [ ]:
columns_to_remove = [
    'label_pol', 'label_sos', 'label_eko', 'label_aman',
    'label_ide', 'label_sda', 'label_demo', 'label_geo'
]

train_dataset_with_embeddings_cleaned = train_dataset_with_embeddings.drop(columns=columns_to_remove)

# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_dataset_with_embeddings_cleaned, test_size=0.2, stratify=train_dataset_with_embeddings['label'], random_state=42)

In [ ]:
train_df_split.drop('label', axis=1).columns

In [ ]:
class_weights_dict = dict()
n, num_class = train_df_split.shape[0], len(train_df_split['label'].unique())
for label in train_df_split['label'].unique():
    n_l = train_df_split[train_df_split['label'] == label].shape[0]
    class_weights_dict[label] = n / (num_class * n_l)

class_weights_dict

{0: 0.1851132326820604,
 1: 1.371299342105263,
 2: 1.9757109004739337,
 3: 2.0236650485436893,
 5: 4.0870098039215685,
 6: 9.474431818181818,
 7: 29.776785714285715,
 4: 2.063737623762376}

In [ ]:
val_df_split

                                                  text  label  \
0    pak prabowo memang menunjukkan dedikasinya ter...      4   
1    ganjar pranowo beberkan syarat terjadinya kerj...      0   
2    testimoni pendeta dari magetan yang datang ke ...      2   
3    kalau anies presiden kebebasan berpendapat dij...      0   
4    capres ganjar pranowo akan memberikan ruang ba...      0   
..                                                 ...    ...   
829  dalam setiap langkahnya warga jabar menunjukka...      0   
830  salut dengan program kerja pak ganjar mahfud s...      0   
831  prabowo pengkajian sumber air perlu agar manfa...      5   
832  dalam setiap tagar jnk bersama ganjar mahfud k...      0   
833  mengurai kontroversi pernyataan anies baswedan...      0   

     pendidikan rendah  kelompok etnis  masyarakat lintas etnis  kaum difabel  \
0                    0               0                        0             0   
1                    0               0                   

In [ ]:
train_df_split

                                                   text  label  \
0     joget joget menyenangkan tapi pemuda lebih mem...      0   
1     ganjar mahfud membangun sinergi antarnegara as...      0   
2     yuk pemilih ganjar monggo di sah kan kosa kata...      1   
3     kontribusi pasangan capres cawapres ganjar pra...      2   
4     suasana optimis melingkupi jawa tengah saat ma...      0   
...                                                 ...    ...   
3330  ganjar pranowo dan mahfud mari bersama kita bu...      0   
3331  untuk petani paslon 03 capres ganjar pranowo s...      2   
3332  breaking news teman kita ditanggkap penyidik b...      0   
3333  koalisi anies ganjar pada awalnya sih malu mal...      0   
3334  ayo ganjar tuntaskan permasalahan yang ada di ...      0   

      pendidikan rendah  kelompok etnis  masyarakat lintas etnis  \
0                     0               0                        0   
1                     0               0                        0   
2  

In [ ]:
numerical_cols = train_df_split.select_dtypes(include=['int64', 'float64']).columns
train_df_split[numerical_cols] = train_df_split[numerical_cols].astype(float)
val_df_split[numerical_cols] = val_df_split[numerical_cols].astype(float)

In [ ]:
train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

train_pool = Pool(
    # data=train_df_split.drop("label", axis=1).to_numpy(),
    data=train_df_split.drop(['label','text'], axis=1).to_numpy(),
    label=train_df_split['label'].to_numpy(),
    feature_names=train_df_split.drop(['label','text'], axis=1).columns.to_list()
)

val_pool = Pool(
    # data=val_df_split.drop("label", axis=1).to_numpy(),
    data=val_df_split.drop(['label','text'], axis=1).to_numpy(),
    label=val_df_split['label'].to_numpy(),
    feature_names=val_df_split.drop(['label','text'], axis=1).columns.to_list()
)

In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights=class_weights_dict,
    iterations = 100,
    learning_rate = 0.01
)

# cb_model.fit(
#     train_pool,
# #     eval_set=val_pool
# )

In [ ]:
cv_method = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
cv_results_model_1 = cross_val_score(estimator=cb_model, X=train_dataset_with_embeddings.drop(["label",'text'], axis=1), y=train_dataset_with_embeddings["label"], cv=cv_method,
                                     verbose=100, scoring='balanced_accuracy')

print("Model 1 - Balanced Accuracy scores: ")
print(cv_results_model_1)
print("Mean: ")
print(cv_results_model_1.mean().round(3))

In [ ]:
# Load the tokenizer from the local directory
local_model_path = 'results/checkpoint-500/'
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

# Define the tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model from the local directory
model = AutoModelForSequenceClassification.from_pretrained(local_model_path, num_labels=8, ignore_mismatched_sizes=True)

In [ ]:
import shutil
shutil.make_archive('results/checkpoint-x', 'zip', 'results/checkpoint-500')

In [ ]:
# Only BERT
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

In [ ]:
train_dataset_with_embeddings

                                                   text  label  \
0     kunjungan prabowo ini untuk meresmikan dan men...      5   
1     anies dapat tepuk tangan meriah saat jadi rekt...      0   
2     emng bener sih pendukung 01 ada yang goblok be...      6   
3     sewaktu anies bersikap kritis ke kinerja pak p...      0   
4     anies baswedan harap aparatur sipil negara ter...      0   
...                                                 ...    ...   
4164  ngeliat debat kemaren pas prabowo kicep kekira...      0   
4165  masyarakat yakin bahwa prabowo gibran memiliki...      0   
4166  imo both are irrational but yang satu jauh leb...      2   
4167  look at that pak ganjar anda sdh berkecimpung ...      3   
4168  acara ini tidak hanya memasak tetapi calon pre...      5   

      pendidikan rendah  kelompok etnis  masyarakat lintas etnis  \
0                     0               0                        0   
1                     0               0                        0   
2  

In [ ]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_dataset_with_embeddings, test_size=0.2, stratify=train_dataset_with_embeddings['label'], random_state=42)

train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

train_pool = Pool(
    # data=train_df_split.drop("label", axis=1).to_numpy(),
    data=train_df_split.drop('label', axis=1).to_numpy(),
    label=train_df_split['label'].to_numpy(),
    text_features=['text'],
    feature_names=train_df_split.drop('label', axis=1).columns.to_list()
)

val_pool = Pool(
    # data=val_df_split.drop("label", axis=1).to_numpy(),
    data=val_df_split.drop('label', axis=1).to_numpy(),
    label=val_df_split['label'].to_numpy(),
    text_features=['text'],
    feature_names=val_df_split.drop('label', axis=1).columns.to_list()
)

In [ ]:
import torch
import gc


# Mengosongkan cache CUDA
torch.cuda.empty_cache()

# Mengumpulkan garbage
gc.collect()

# Memeriksa penggunaan memori CUDA
print(f"Memory Reserved: {torch.cuda.memory_reserved()} bytes")
print(f"Memory Allocated: {torch.cuda.memory_allocated()} bytes")
print(f"Max Memory Reserved: {torch.cuda.max_memory_reserved()} bytes")

# Restart kernel/environment jika diperlukan

In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights=class_weights_dict,
    learning_rate = 0.01
)

cb_model.fit(
    train_pool,
    eval_set=val_pool
)

0:	learn: 0.6041802	test: 0.3541278	best: 0.3541278 (0)	total: 62.5ms	remaining: 1m 2s
100:	learn: 0.8450560	test: 0.4571283	best: 0.4571684 (96)	total: 3.26s	remaining: 29s
200:	learn: 0.8735332	test: 0.4602418	best: 0.4746669 (188)	total: 6.55s	remaining: 26s
300:	learn: 0.8924493	test: 0.4560045	best: 0.4746669 (188)	total: 9.89s	remaining: 23s
400:	learn: 0.9024723	test: 0.4560181	best: 0.4746669 (188)	total: 13s	remaining: 19.4s
500:	learn: 0.9115117	test: 0.4606953	best: 0.4746669 (188)	total: 15.9s	remaining: 15.9s
600:	learn: 0.9167444	test: 0.4619494	best: 0.4746669 (188)	total: 18.7s	remaining: 12.4s
700:	learn: 0.9233699	test: 0.4596752	best: 0.4746669 (188)	total: 21.4s	remaining: 9.11s
800:	learn: 0.9275399	test: 0.4545974	best: 0.4746669 (188)	total: 24s	remaining: 5.96s
900:	learn: 0.9298129	test: 0.4549175	best: 0.4746669 (188)	total: 26.6s	remaining: 2.93s
999:	learn: 0.9321927	test: 0.4576122	best: 0.4746669 (188)	total: 29.2s	remaining: 0us
bestTest = 0.4746669241
be

In [ ]:
# text + embedding
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5151185988709432
